# Train π0 (`pi0`) on RunPod — FR5 pick-and-place — **self-contained**

π0 is a flow-matching VLA: a PaliGemma-2B vision-language model conditions a 300M-parameter action expert that predicts 50-step action chunks via flow matching.

This notebook is **fully standalone**: no git clone, no repo files. The dataset class,
the π0 wrapper, and the training loop are all defined in cells below (ported from the
`fairino-fr5-policies` repo so **checkpoints stay byte-compatible with the repo's
`common/deploy.py`** on the robot box).

**Finetuning recipe:** pretrained `lerobot/pi0_base` weights (verified load with a hard
key-match check) + **LoRA adapters on the frozen VLM** + **full training of the 300M
action expert & projections**.

## Pod setup
| | |
|---|---|
| **Template** | **Python 3.12** + CUDA ≥ 12.1 GPU pod. lerobot 0.5.1 requires 3.12 — pick an image on an **Ubuntu 24.04** base (that's what ships 3.12); RunPod's default 22.04 PyTorch images are 3.11. Verify in the pod terminal with `python --version`. |
| **GPU** | 48 GB (A40 / A6000 / L40S) recommended · 24 GB (4090) works with `expert_only` · 80 GB = full finetune |
| **Disk** | ≥ 60 GB (~5 GB PaliGemma + dataset + checkpoints, ~5 GB each) |

## One-time prerequisites
1. **HF token** (read + write) whose account has **accepted the PaliGemma license**:
   <https://huggingface.co/google/paligemma-3b-pt-224> — gated; the tokenizer comes from there.
   The model weights themselves come from **`lerobot/pi0_base`** (~6 GB, the openpi port on the
   lerobot hub) — this is what makes it *finetuning*; without it lerobot random-inits the
   whole model.
2. **Convert raw episodes → LeRobot, then push to the Hub.** Your recordings are raw
   `episode_XXXX/` folders (data.csv + wrist_cam.mp4 + scene_cam.mp4 + meta.json), e.g. on
   the Hub as `Slifold/episodes_20260717`. **Easiest: run
   `notebooks/convert_and_push_dataset.ipynb`** — it pulls the raw set, converts, and pushes
   the LeRobot dataset; then set `HF_DATASET_REPO` below to that output repo. Manual path:
   First convert them with the repo's converter (produces a 7-D joint state + 7-D action,
   wrist-cam only, 30 fps, and a default task string when meta.json's instruction is empty):
   ```bash
   python common/convert_episodes.py --episodes episodes --out lerobot_dataset --extract-frames
   ```
   Then push the resulting `lerobot_dataset/` to the Hub:
   ```bash
   pip install huggingface_hub && huggingface-cli login
   python -c "from huggingface_hub import HfApi; api=HfApi(); \
       api.create_repo('<you>/fr5-pick-place-lerobot', repo_type='dataset', private=True, exist_ok=True); \
       api.upload_folder(folder_path='lerobot_dataset', repo_id='<you>/fr5-pick-place-lerobot', repo_type='dataset')"
   ```
   (or `python tools/push_dataset_hf.py --repo <you>/fr5-pick-place-lerobot` if you have the repo).

## 1 · Parameters

| variable | meaning |
|---|---|
| `FINETUNE_MODE` | `auto` picks by VRAM · **`lora`** = LoRA adapters on the frozen VLM **+ full action-expert training** (the recommended recipe) · `full` = finetune everything · `freeze_vision` = freeze SigLIP only · `expert_only` = freeze the whole VLM, no adapters |
| `BATCH_SIZE` | `None` = auto by VRAM |
| `PROPRIO_MODE` | `full` / `dropout` / `none` — proprioception benchmark axis |

**VRAM cheat-sheet** (bf16 + gradient checkpointing always on): 24 GB → `expert_only`, batch 2 · 48 GB → `lora`, batch 4 · 80 GB → `lora` batch 8 (or `full`).

> `lora` adapts the VLM to FR5 visuals with only ~7M extra trainable params while the
> 300M action expert trains fully — the best fit for 150 episodes. `expert_only` is the
> low-VRAM fallback (VLM completely untouched).

In [ ]:
import os

HF_TOKEN        = os.environ.get("HF_TOKEN", "")        # hf_... (read+write, PaliGemma licence accepted)
HF_DATASET_REPO = "<you>/fr5-pick-place-lerobot"

USE_WANDB     = True                                     # auto-disables if no API key
WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")      # wandb.ai/authorize
WANDB_PROJECT = "fr5-vla-benchmark"

POLICY       = "pi0"
PRETRAINED   = "lerobot/pi0_base"  # "" -> random init (smoke tests only — NOT finetuning)
TASK_TEXT    = "pick up the block and place it in the bin"
FINETUNE_MODE = "auto"      # auto | lora | full | freeze_vision | expert_only
QUANTIZE     = "nf4"        # none | nf4 | int8 — k-bit quantization of the FROZEN VLM.
                            # "nf4" (QLoRA) stores the 2B VLM as 4-bit normal-float:
                            # ~4.6 GB -> ~1.4 GB, so a bigger batch fits. Matmuls still
                            # run in bf16, so speed is close to unquantized. Needs LoRA
                            # (4-bit weights take no gradient) — the adapters + the
                            # action expert carry all the learning. Set "none" for a
                            # plain bf16 run when VRAM is not the constraint.
LORA_RANK    = 16           # LoRA rank on the VLM q/k/v/o (lora mode)
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
BATCH_SIZE   = None         # None -> picked from GPU VRAM
MAX_EPOCHS   = 100
LR           = 2.5e-5
WEIGHT_DECAY = 0.01
GRAD_CLIP    = 1.0
CHUNK_SIZE   = 50           # action horizon (pi0 default, 1.67 s @ 30 Hz)
FRAME_STRIDE = 5            # sample every Kth frame as a chunk START (cuts redundant 30fps samples; chunks stay full-rate)
VAL_FRAC     = 0.1
AUG_LEVEL    = "crops"      # none | crops   (train-time image augmentation)
PROPRIO_MODE = "full"       # full | dropout | none
LOG_EVERY    = 25           # optimizer STEPS between metric logs (wandb + metrics_steps.csv).
                            # Averaged over the window: a raw per-step loss at batch 8 is
                            # mostly sampling noise. Lower = finer curve, more IO.
SAVE_EVERY   = 10           # epochs between periodic checkpoints
PUSH_EVERY   = 10           # epochs between Hub pushes of best.pt (0 = only the
                            # final cell). /workspace dies with the pod, so a
                            # checkpoint that exists only here is one crash from gone.
SEED         = 42

WORK     = "/workspace"
DATA_DIR = f"{WORK}/lerobot_dataset"
CKPT_DIR = f"{WORK}/checkpoints_{POLICY}"

# never hardcode the token. Preferred: set it as a pod secret / env var
#   (RunPod: Pod -> Edit -> Environment Variables -> HF_TOKEN = hf_...).
# Fallback: if it isn't in the environment, prompt for it here (masked input,
# not saved into the notebook).
if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass("HF token (hf_...): ").strip()
assert HF_TOKEN.startswith("hf_"), "no HF token — PaliGemma + the pretrained weights are gated"
print("parameters set")

## 2 · Install dependencies

`lerobot==0.5.1` pins the torch/transformers stack — ~3–5 min on first run.
(pip may replace the pod's preinstalled torch; that's expected.)

In [ ]:
import sys, subprocess, importlib

if sys.version_info < (3, 12):
    raise SystemExit(
        f"This pod is Python {sys.version_info.major}.{sys.version_info.minor}, but lerobot "
        "0.5.1 needs >= 3.12. Deploy the pod with a Python 3.12 image (an Ubuntu 24.04 "
        "base ships 3.12) and re-run. Quick check in the pod terminal: `python --version`.")

# Deliberately NOT quiet (-q). A pod install that silently resolves the wrong
# version costs far more to debug an hour into training than this log costs to
# scroll past — and pip's resolver messages are the only warning you get.
# torch / torchvision are NOT installed here: the pod image ships a build matched
# to its CUDA driver, and pip would happily replace it with a mismatched one.
PKGS = [
    "lerobot==0.5.1",               # PI policy implementation + pinned DL stack
    "transformers==5.9.0",          # pin: newer breaks lerobot's groot import
    "peft>=0.17",                   # LoRA adapters on the VLM
    "bitsandbytes>=0.43",           # NF4/int8 quantization of the frozen VLM (QLoRA)
    "accelerate>=0.34",             # device/dtype plumbing peft + bitsandbytes rely on
    "safetensors>=0.4",             # pretrained-checkpoint format
    "huggingface_hub>=0.34",        # dataset pull + weight download + hub push
    "opencv-python-headless>=4.9",  # cv2: frame decode. headless = no GUI libs, which a pod lacks
    "wandb>=0.17",                  # experiment tracking
    "numpy>=2.0,<2.3",              # lerobot is not yet numpy 2.3 clean
    "pandas", "pyarrow",            # dataset parquet handling
    "matplotlib", "tqdm"]           # plots + progress bars
subprocess.run([sys.executable, "-m", "pip", "install", *PKGS], check=True)

# Import EVERYTHING the notebook will touch, right now. A missing or mismatched
# dependency then fails here in seconds, instead of after a multi-GB dataset pull
# and halfway through building a 3.5B-parameter model.
_missing = []
for _m in ['torch', 'torchvision', 'lerobot', 'transformers', 'peft', 'bitsandbytes', 'cv2', 'numpy', 'pandas', 'pyarrow', 'safetensors', 'matplotlib', 'wandb', 'huggingface_hub', 'tqdm']:
    try:
        importlib.import_module(_m)
    except Exception as _e:
        _missing.append(f"{_m}: {type(_e).__name__}: {_e}")
if _missing:
    raise SystemExit("dependency check FAILED — fix these before continuing:\n  "
                     + "\n  ".join(_missing))

import torch, lerobot, transformers, numpy, wandb
import peft, bitsandbytes

print("\nall imports OK · " + " · ".join([
    f"torch {torch.__version__}", f"lerobot {lerobot.__version__}",
    f"transformers {transformers.__version__}",
    f"peft {peft.__version__}", f"bitsandbytes {bitsandbytes.__version__}",

    f"numpy {numpy.__version__}", f"wandb {wandb.__version__}"]))


## 3 · HuggingFace auth + gated-weights check

Fails fast with the license URL if the token can't access PaliGemma.

In [ ]:
from huggingface_hub import login, whoami, auth_check
from huggingface_hub.errors import GatedRepoError

login(token=HF_TOKEN, add_to_git_credential=False)
print("logged in as:", whoami()["name"])

try:
    auth_check("google/paligemma-3b-pt-224")
    print("PaliGemma licence OK — gated weights accessible")
except GatedRepoError:
    raise SystemExit("PaliGemma is gated for this token — accept the licence at "
                     "https://huggingface.co/google/paligemma-3b-pt-224 and re-run")

### 3b · Weights & Biases login

Optional but recommended — live loss curves, run comparison across policies /
finetune modes, and the GT-vs-prediction figure land in one dashboard.
No key → the notebook silently falls back to local `metrics.csv` + matplotlib only.

In [ ]:
import wandb

if USE_WANDB and WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY)
    print("wandb: logged in — project", WANDB_PROJECT)
elif USE_WANDB:
    USE_WANDB = False
    print("wandb: no WANDB_API_KEY set -> disabled (metrics.csv still written)")
else:
    print("wandb: disabled")

## 4 · GPU check → resolve memory mode + batch size

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch

assert torch.cuda.is_available(), "no CUDA GPU — pick a GPU pod"
name = torch.cuda.get_device_name(0)
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
vram_free  = torch.cuda.mem_get_info()[0] / 1e9
# Size everything from FREE VRAM, not total. A crashed kernel holds its CUDA context
# and allocations until the pod restarts, and inside a container those PIDs usually
# cannot be killed — so "48 GB card" can mean 21 GB you can actually use, and a batch
# size chosen from the nameplate number OOMs on the first step.
vram = min(vram_total, vram_free)
assert torch.cuda.is_bf16_supported(), "bf16 unsupported — use an Ampere or newer GPU"
print(f"{name}  {vram_total:.0f} GB total · {vram_free:.0f} GB free")

if QUANTIZE == "nf4":
    assert torch.cuda.get_device_capability(0) >= (7, 5), (
        "NF4 needs compute capability >= 7.5 (Turing+). Set QUANTIZE = 'none'.")

if FINETUNE_MODE == "auto":
    # NF4 shrinks the frozen VLM from ~4.6 GB to ~1.4 GB, so QLoRA fits on cards where
    # a bf16 LoRA run would not — without it, a sub-40 GB card can only afford the
    # expert-only path (VLM frozen AND unadapted).
    FINETUNE_MODE = "lora" if (vram >= 40 or QUANTIZE in ("nf4", "int8")) else "expert_only"
if FINETUNE_MODE == "full" and QUANTIZE != "none":
    raise SystemExit(f"FINETUNE_MODE='full' cannot train a {QUANTIZE}-quantized VLM "
                     "(4-bit weights take no gradient). Use 'lora', or QUANTIZE='none'.")
if BATCH_SIZE is None:
    # sized from measured usage: pi0 LoRA is ~21 GB at batch 4, ~1.8 GB/extra sample,
    # so 40 GB fits ~8 and 70 GB+ fits ~16. Adjust BATCH_SIZE above if you OOM.
    BATCH_SIZE = {"expert_only": 2 if vram < 40 else 4,
                  "lora":          16 if vram >= 70 else (8 if vram >= 40 else 4),
                  "freeze_vision": 8 if vram >= 40 else 4,
                  "full":          8 if vram >= 70 else (4 if vram >= 40 else 2)}[FINETUNE_MODE]
    if QUANTIZE == "nf4" and FINETUNE_MODE == "lora":
        BATCH_SIZE += 2          # ~3.2 GB of weights freed / ~1.8 GB per extra sample
print(f"finetune_mode = {FINETUNE_MODE}   quantize = {QUANTIZE}   batch_size = {BATCH_SIZE}")
# VRAM you don't have is the single most common cause of a failed start here. Two
# very different causes, and they need opposite responses — so report, never act:
#   • ANOTHER LIVE JOB (a second notebook, a training you left running). Normal and
#     legitimate. This run has to fit in what's left, which is what sizing from
#     vram_free above does. Expect both jobs to be slower — they share SMs too.
#   • A CRASHED KERNEL holding its CUDA context. Only a restart reliably clears it;
#     inside a container the PIDs nvidia-smi prints are usually host PIDs you have
#     no permission to signal anyway.
# NEVER blanket-kill compute PIDs to free VRAM: from in here the two cases are
# indistinguishable, and guessing wrong destroys a running job. Identify it first.
if vram_free < 0.7 * vram_total:
    print(f"\nNOTE: {vram_free:.0f} of {vram_total:.0f} GB free — something else holds "
          f"{vram_total - vram_free:.0f} GB.\n"
          f"      Check what it is before doing anything about it:  nvidia-smi\n"
          f"      If it's a live job, this run is sized to fit alongside it (above).\n"
          f"      If it's a dead kernel, restart the pod — that is the reliable clear.")

## 5 · Pull the dataset from the Hub

In [ ]:
from huggingface_hub import snapshot_download
import subprocess, sys, json, pathlib, urllib.request

# Pull whatever HF_DATASET_REPO points at, into a FIXED dir (re-run safe).
DL_DIR = f"{WORK}/hf_dataset"
snapshot_download(HF_DATASET_REPO, repo_type="dataset", local_dir=DL_DIR)

# Already LeRobot (has meta/info.json)? use it. RAW episodes (episode_*/ folders)?
# convert to LeRobot on the pod. DATA_DIR is derived fresh each run -> idempotent.
if pathlib.Path(DL_DIR, "meta", "info.json").exists():
    DATA_DIR = DL_DIR
else:
    DATA_DIR = f"{WORK}/fr5_lerobot"
    if not pathlib.Path(DATA_DIR, "meta", "info.json").exists():
        print("raw episodes detected -> converting to LeRobot (wrist + scene)...")
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/SreevaatsavB/fairino-fr5-policies/main/common/convert_episodes.py",
            "convert_episodes.py")
        subprocess.run([sys.executable, "convert_episodes.py",
                        "--episodes", DL_DIR, "--out", DATA_DIR,
                        "--extract-frames", "--cameras", "wrist,scene"], check=True)

info = json.loads(pathlib.Path(DATA_DIR, "meta", "info.json").read_text())
STATE_DIM  = info["features"]["observation.state"]["shape"][0]
ACTION_DIM = info["features"]["action"]["shape"][0]
CAMERAS    = [k for k in info["features"] if k.startswith("observation.images.")]
CAMERA_NAMES = [k.split(".")[-1] for k in CAMERAS]
print(f"episodes = {info['total_episodes']}   frames = {info['total_frames']}   fps = {info['fps']}")
print(f"state_dim = {STATE_DIM}   action_dim = {ACTION_DIM}   cameras = {CAMERA_NAMES}   ->  {DATA_DIR}")
assert "wrist_cam" in CAMERA_NAMES, f"expected a wrist camera; got {CAMERA_NAMES}"

### 5b · Eyeball the data

A wrist-cam frame and one episode's 7-D action traces — if these look wrong
(black frames, flat traces), stop and fix the dataset before spending GPU-hours.

In [ ]:
import pandas as pd, pathlib
import matplotlib.pyplot as plt, matplotlib.image as mpimg
import pyarrow.parquet as pq

EP_VIEW = 0        # <-- change to preview any episode index

df_raw = pd.read_parquet(pathlib.Path(DATA_DIR, "data/chunk-000/file-000.parquet"))
ep = df_raw[df_raw.episode_index == EP_VIEW]
tasks = pq.read_table(pathlib.Path(DATA_DIR, "meta/tasks.parquet")).to_pandas()
task = tasks.loc[tasks.task_index == int(ep["task_index"].iloc[0]), "task"].iloc[0]

fig, ax = plt.subplots(1, len(CAMERAS) + 1, figsize=(5 * (len(CAMERAS) + 1), 3.5))
for a, cam in zip(ax, CAMERAS):
    fr = sorted(pathlib.Path(DATA_DIR, "frames", cam, f"ep-{EP_VIEW:03d}").glob("*.jpg"))
    if fr:
        a.imshow(mpimg.imread(fr[len(fr)//2])); a.axis("off")
        a.set_title(f"{cam.split('.')[-1]} ({len(fr)} frames)")
pd.DataFrame(ep["action"].tolist()).plot(ax=ax[-1], legend=False,
    title=f"ep{EP_VIEW} actions (7-D, {len(ep)} steps)")
plt.suptitle(f"episode {EP_VIEW}  —  {task!r}", fontsize=11)
plt.tight_layout(); plt.show()

## 6 · Dataset class (inline)

Ported from the repo's `common/dataset.py`: chunked action targets with padding,
episode-level train/val split (no leakage), ImageNet-normalised 224×224 frames,
and optional UMI-style crop/colour-jitter augmentation for training.

In [ ]:
import json, random
from pathlib import Path

import cv2
import numpy as np
import pyarrow.parquet as pq
import torch
from torch.utils.data import Dataset
from torchvision import transforms

_IMAGENET_MEAN = [0.485, 0.456, 0.406]
_IMAGENET_STD  = [0.229, 0.224, 0.225]


def _build_transform(image_size, aug_level):
    h, w = image_size
    if aug_level == "none":
        return transforms.Compose([
            transforms.Resize(image_size),
            transforms.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD)])
    if aug_level == "crops":   # UMI-calibrated jitter
        return transforms.Compose([
            transforms.Resize((int(h * 1.12), int(w * 1.12))),
            transforms.RandomCrop(image_size),
            transforms.ColorJitter(brightness=0.3, contrast=0.4, saturation=0.5, hue=0.08),
            transforms.RandomGrayscale(p=0.05),
            transforms.Normalize(mean=_IMAGENET_MEAN, std=_IMAGENET_STD)])
    raise ValueError(f"unknown aug_level {aug_level!r}")


class FR5Dataset(Dataset):
    """LeRobot-v3 FR5 episodes -> (state, action chunk, pad mask, image, task)."""

    def __init__(self, root, chunk_size=50, image_size=(224, 224),
                 episode_indices=None, aug_level="none", frame_stride=1):
        self.root, self.chunk_size, self.image_size = Path(root), chunk_size, image_size
        self.frame_stride = max(1, int(frame_stride))
        self.info = json.loads((self.root / "meta/info.json").read_text())
        self.camera_keys = [k for k in self.info.get("features", {})
                            if k.startswith("observation.images.")] or                            ["observation.images.wrist_cam"]
        self.df = pq.read_table(self.root / "data/chunk-000/file-000.parquet").to_pandas()
        self.episodes = pq.read_table(
            self.root / "meta/episodes/chunk-000/file-000.parquet").to_pandas()
        if episode_indices is not None:
            self.episodes = self.episodes[
                self.episodes["episode_index"].isin(episode_indices)].reset_index(drop=True)
        self._samples = [(int(e.episode_index), t)
                         for _, e in self.episodes.iterrows()
                         for t in range(int(e.dataset_from_index), int(e.dataset_to_index), self.frame_stride)]
        tasks = self.root / "meta/tasks.parquet"
        self._task_map = (dict(zip(*pq.read_table(tasks).to_pandas()
                                   [["task_index", "task"]].T.values.tolist()))
                          if tasks.exists() else {})
        self._tf = _build_transform(image_size, aug_level)

    def __len__(self): return len(self._samples)

    def __getitem__(self, idx):
        ep_idx, frame_abs = self._samples[idx]
        row = self.df.iloc[frame_abs]
        ep = self.episodes[self.episodes.episode_index == ep_idx].iloc[0]
        ep_to = int(ep.dataset_to_index)

        state = torch.tensor(row["observation.state"], dtype=torch.float32)
        chunk = self.df.iloc[frame_abs:min(frame_abs + self.chunk_size, ep_to)]
        actions = torch.tensor(np.array(chunk["action"].tolist()), dtype=torch.float32)
        pad = self.chunk_size - len(actions)
        is_pad = torch.zeros(self.chunk_size, dtype=torch.bool)
        if pad > 0:
            actions = torch.cat([actions, actions[-1:].expand(pad, -1)])
            is_pad[-pad:] = True

        sample = {"observation.state": state, "action": actions, "action_is_pad": is_pad,
                  "task": self._task_map.get(int(row.get("task_index", 0)), TASK_TEXT)}
        for cam in self.camera_keys:                   # load EVERY camera (wrist + scene)
            jpg = self.root / "frames" / cam / f"ep-{ep_idx:03d}" / f"{int(row.frame_index):06d}.jpg"
            frame = cv2.imread(str(jpg))
            if frame is None:                          # video fallback
                cap = cv2.VideoCapture(str(self.root / "videos" / cam / "chunk-000" /
                                           f"file-{ep_idx:03d}.mp4"))
                cap.set(cv2.CAP_PROP_POS_FRAMES, int(row.frame_index))
                ok, frame = cap.read(); cap.release()
                assert ok, f"missing {cam} frame ep{ep_idx} idx{int(row.frame_index)}"
            img = torch.from_numpy(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                                   ).permute(2, 0, 1).float() / 255.0
            sample[cam] = self._tf(img)
        return sample

    def get_stats(self):
        sub = self.df[self.df.episode_index.isin(self.episodes.episode_index.tolist())]
        s, a = (np.array(sub[k].tolist()) for k in ("observation.state", "action"))
        return {"state_mean": s.mean(0).astype(np.float32),
                "state_std":  s.std(0).clip(1e-6).astype(np.float32),
                "state_min":  s.min(0).astype(np.float32),
                "state_max":  s.max(0).astype(np.float32),
                "action_mean": a.mean(0).astype(np.float32),
                "action_std":  a.std(0).clip(1e-6).astype(np.float32),
                "action_min":  a.min(0).astype(np.float32),
                "action_max":  a.max(0).astype(np.float32)}

    @staticmethod
    def episode_split(n_episodes, val_frac=0.1, seed=42):
        idx = list(range(n_episodes))
        random.seed(seed); random.shuffle(idx)
        n_val = max(1, int(val_frac * n_episodes))
        return idx[n_val:], idx[:n_val]

print("FR5Dataset defined")

## 6b · Replay a training episode as a GIF

Data inspection, no model involved — run this before committing to a long training run. Shows every camera next to the recorded joint trajectories (measured state solid, commanded action dashed) with a playhead, and prints the episode's language prompt so you can confirm the per-episode text actually landed in the dataset. Saves a real `.gif` you can download.

In [ ]:
# ── replay a TRAINING episode as a GIF: cameras + recorded trajectories ───────
GIF_EPISODE = 0     # which episode to look at
GIF_STRIDE  = 2     # show every Nth frame. Free to stride here: this is pure data
                    # inspection, with no action queue to desync (unlike the 12b replay)
GIF_MAX     = 150   # frame cap — GIF size grows linearly with it
GIF_FPS     = 12
GIF_PATH    = f"{WORK}/episode_{GIF_EPISODE:03d}.gif"

import pathlib
import numpy as np, matplotlib.pyplot as plt
from matplotlib import animation, gridspec
from IPython.display import Image, display
from tqdm.auto import tqdm

# aug_level="none": augmentation would show you jitter that the stored data doesn't
# have. This is the raw episode as recorded, which is the point.
ds   = FR5Dataset(DATA_DIR, chunk_size=CHUNK_SIZE, episode_indices=[GIF_EPISODE],
                  aug_level="none", frame_stride=GIF_STRIDE)
n    = min(len(ds), GIF_MAX)
cams = ds.camera_keys
_mean = np.array(_IMAGENET_MEAN).reshape(3, 1, 1)
_std  = np.array(_IMAGENET_STD).reshape(3, 1, 1)

frames, state, action, task = [], [], [], None
for i in tqdm(range(n), desc="loading frames"):
    it = ds[i]
    frames.append(np.hstack([np.clip(it[c].numpy() * _std + _mean, 0, 1).transpose(1, 2, 0)
                             for c in cams]))          # undo the ImageNet norm
    state.append(it["observation.state"].numpy())
    action.append(it["action"][0].numpy())             # first action of the chunk
    task = task or it["task"]
state, action = np.asarray(state), np.asarray(action)

print(f"episode {GIF_EPISODE}: {len(ds)} frames at stride {GIF_STRIDE} -> {n} shown")
print(f"prompt: {task!r}")          # per-episode language, straight from meta/tasks.parquet

D      = state.shape[1]
joints = list(range(min(6, D)))
grip   = D - 1 if D == 7 else None

fig = plt.figure(figsize=(12, 4.4), dpi=60)     # dpi is the other GIF-size knob
gs  = gridspec.GridSpec(2, 2, width_ratios=[1.25, 1], height_ratios=[2.4, 1],
                        hspace=0.32, wspace=0.16)
ax_img = fig.add_subplot(gs[:, 0]); ax_img.axis("off")
ax_j   = fig.add_subplot(gs[0, 1])
ax_g   = fig.add_subplot(gs[1, 1])

im = ax_img.imshow(frames[0])
ax_img.set_title(" | ".join(c.split(".")[-1] for c in cams) + f"   —   {task[:60]}", fontsize=8)

colors = plt.cm.tab10(np.linspace(0, 1, 10))
for j in joints:                       # state solid, commanded action dashed
    ax_j.plot(state[:, j],  color=colors[j], lw=1.3, label=f"j{j+1}")
    ax_j.plot(action[:, j], color=colors[j], lw=1.0, ls="--", alpha=0.8)
ax_j.set_ylabel("joint"); ax_j.grid(alpha=0.25)
ax_j.legend(fontsize=6, ncol=6, loc="upper right")
ax_j.set_title("— measured state    -- commanded action", fontsize=8)

if grip is not None:
    ax_g.plot(state[:, grip],  color="k", lw=1.3)
    ax_g.plot(action[:, grip], color="k", lw=1.0, ls="--", alpha=0.8)
    ax_g.fill_between(range(n), state[:, grip], color="k", alpha=0.12)
    ax_g.set_ylabel("gripper")
ax_g.set_xlabel("frame"); ax_g.grid(alpha=0.25)

heads = [a.axvline(0, color="crimson", lw=1.3) for a in (ax_j, ax_g)]

def _update(i):
    im.set_data(frames[i])
    for h in heads:
        h.set_xdata([i, i])
    return [im, *heads]

anim = animation.FuncAnimation(fig, _update, frames=n, interval=1000 / GIF_FPS, blit=False)
# PillowWriter writes a real .gif with no ffmpeg dependency — pods often lack it.
anim.save(GIF_PATH, writer=animation.PillowWriter(fps=GIF_FPS))
plt.close(fig)

mb = pathlib.Path(GIF_PATH).stat().st_size / 1e6
print(f"saved {GIF_PATH}  ({mb:.1f} MB)"
      + ("   <- shrink with GIF_MAX / GIF_STRIDE / dpi" if mb > 20 else ""))
display(Image(filename=GIF_PATH))


## 7 · π0 policy wrapper (inline)

Ported from the repo's `policies/pi0/model.py` — identical normalisation, batch
building, proprio masking, and config threading, wrapping lerobot's `PI0Policy`.
The checkpoint this produces loads directly in the repo's `common/deploy.py`.

In [ ]:
from contextlib import contextmanager
from dataclasses import dataclass

import torch
import torch.nn as nn
from transformers import AutoTokenizer

# lerobot eagerly imports its groot policy in policies/__init__, whose config has
# a dataclass bug (GR00TN15Config) that crashes the whole import on some builds.
# We don't use groot -> stub it in sys.modules before importing any lerobot policy.
import sys as _sys, types as _types
for _m in ("lerobot.policies.groot", "lerobot.policies.groot.configuration_groot"):
    _sys.modules.setdefault(_m, _types.ModuleType(_m))
_sys.modules["lerobot.policies.groot.configuration_groot"].GrootConfig = None

from lerobot.policies.pi0.configuration_pi0 import PI0Config as _LRConfig
from lerobot.policies.pi0.modeling_pi0 import PI0Policy

# lerobot's pi0-family passes a Long attention mask to torch.where, which
# PyTorch >= 2.8 rejects (needs bool). Wrap the method to cast the mask first.
import lerobot.policies.pi0.modeling_pi0 as _pimod
if not getattr(_pimod.PI0Pytorch._prepare_attention_masks_4d, '_bool_patched', False):
    _orig_mask4d = _pimod.PI0Pytorch._prepare_attention_masks_4d
    def _mask4d_bool(self, att_2d_masks, *a, **k):
        return _orig_mask4d(self, att_2d_masks.bool(), *a, **k)
    _mask4d_bool._bool_patched = True
    _pimod.PI0Pytorch._prepare_attention_masks_4d = _mask4d_bool
from lerobot.configs.types import PolicyFeature, FeatureType, NormalizationMode

STATE_KEY, IMAGE_KEY, ACTION_KEY = ("observation.state",
                                    "observation.images.wrist_cam", "action")
LANG_TOKENS    = "observation.language.tokens"
LANG_ATTN_MASK = "observation.language.attention_mask"
_IMN_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_IMN_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def mask_state(state, mode, rate, training):
    """Proprio modes: full = untouched · none = always zeroed ·
    dropout = per-sample zeroed with prob `rate` during training only."""
    if mode == "none":
        return torch.zeros_like(state)
    if mode == "dropout" and training:
        keep = (torch.rand(state.shape[0], 1, device=state.device) >= rate)
        return state * keep.to(state.dtype)
    return state


@dataclass
class PolicyCfg:
    state_dim:  int = 7
    action_dim: int = 7
    chunk_size: int = 50
    use_image:  bool = True
    num_inference_steps: int = 10
    max_state_dim:  int = 32
    max_action_dim: int = 32
    paligemma_variant:     str = "gemma_2b"
    action_expert_variant: str = "gemma_300m"
    tokenizer_max_length:  int = 48
    pretrained:            str = "lerobot/pi0_base"   # "" -> random init (smoke only)
    vlm_lora_rank:         int = 0                  # >0 -> LoRA on VLM q/k/v/o
    vlm_lora_alpha:        int = 32
    vlm_lora_dropout:      float = 0.05
    dtype:                  str  = "bfloat16"
    gradient_checkpointing: bool = True
    freeze_vision_encoder:  bool = False
    train_expert_only:      bool = False
    quantize:               str  = "none"   # none | nf4 | int8 (frozen VLM only)
    camera_names:         tuple = ("wrist_cam", "scene_cam")
    proprio_mode:         str   = "full"
    proprio_dropout_rate: float = 0.3


def _lerobot_config(cfg):
    feats = {STATE_KEY: PolicyFeature(type=FeatureType.STATE, shape=(cfg.state_dim,))}
    norm  = {"STATE": NormalizationMode.IDENTITY, "ACTION": NormalizationMode.IDENTITY}
    if cfg.use_image:
        for _k in [f"observation.images.{c}" for c in cfg.camera_names]:
            feats[_k] = PolicyFeature(type=FeatureType.VISUAL, shape=(3, 224, 224))
        norm["VISUAL"] = NormalizationMode.IDENTITY
    return _LRConfig(
        n_obs_steps=1, chunk_size=cfg.chunk_size, n_action_steps=cfg.chunk_size,
        input_features=feats,
        output_features={ACTION_KEY: PolicyFeature(type=FeatureType.ACTION,
                                                   shape=(cfg.action_dim,))},
        normalization_mapping=norm,
        paligemma_variant=cfg.paligemma_variant,
        action_expert_variant=cfg.action_expert_variant,
        max_state_dim=cfg.max_state_dim, max_action_dim=cfg.max_action_dim,
        num_inference_steps=cfg.num_inference_steps,
        tokenizer_max_length=cfg.tokenizer_max_length,
        dtype=cfg.dtype, gradient_checkpointing=cfg.gradient_checkpointing,
        freeze_vision_encoder=cfg.freeze_vision_encoder,
        # with LoRA the base VLM must be frozen — adapters carry the VLM update
        train_expert_only=cfg.train_expert_only or cfg.vlm_lora_rank > 0)


def _load_pretrained_weights(policy, repo_id):
    """Load openpi-ported weights with version-proof key remapping + a HARD check.

    lerobot's own from_pretrained loads with strict=False and only prints missing
    keys — under transformers >= 5.4 (which dropped the `.vision_model` nesting
    inside SigLIP) that silently leaves the ENTIRE vision tower random-init."""
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file

    sd = load_file(hf_hub_download(repo_id, "model.safetensors"))
    sd = policy._fix_pytorch_state_dict_keys(sd, policy.config)
    sd = {(k if k.startswith("model.") else f"model.{k}"): v for k, v in sd.items()}
    model_keys = set(policy.state_dict().keys())
    if (any(".vision_tower.vision_model." in k for k in sd)
            and not any(".vision_tower.vision_model." in k for k in model_keys)):
        sd = {k.replace(".vision_tower.vision_model.", ".vision_tower."): v
              for k, v in sd.items()}
    missing, unexpected = policy.load_state_dict(sd, strict=False)
    n_loaded = len(model_keys) - len(missing)
    print(f"pretrained load: {n_loaded}/{len(model_keys)} tensors from {repo_id} "
          f"({len(unexpected)} unexpected ignored)")
    if n_loaded < 0.99 * len(model_keys):
        raise RuntimeError(
            f"only {n_loaded}/{len(model_keys)} tensors matched {repo_id} — a partial "
            f"load silently finetunes random weights. First missing: {sorted(missing)[:5]}")


def _inject_vlm_lora(policy, rank, alpha, dropout):
    """LoRA adapters on the (frozen) VLM's attention projections, in place —
    peft's inject_adapter_in_model keeps lerobot's module paths intact.
    Adapter params stay fp32 for stable AdamW on bf16 base weights."""
    from peft import LoraConfig, inject_adapter_in_model

    inject_adapter_in_model(
        LoraConfig(r=rank, lora_alpha=alpha, lora_dropout=dropout,
                   target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], bias="none"),
        policy.model.paligemma_with_expert.paligemma)
    n_lora = 0
    for n, p in policy.named_parameters():
        if "lora_" in n:
            p.data = p.data.float(); p.requires_grad_(True); n_lora += p.numel()
    n_train = sum(p.numel() for p in policy.parameters() if p.requires_grad)
    print(f"LoRA r={rank} on VLM q/k/v/o: {n_lora/1e6:.1f}M adapter params; "
          f"total trainable {n_train/1e6:.0f}M (frozen base VLM + full action expert)")


@contextmanager
def build_context(device, dtype=None):
    """Construct the policy directly on `device` — never staged through host RAM.

    lerobot builds the policy wherever torch's default device points (CPU), then
    `.to(config.device)` at the end. For pi0.5 that means ~3.5B params materialise as
    fp32 in HOST RAM first — ~14 GB of allocation churn that the OOM-killer ends
    ("the kernel appears to have died") on a container whose cgroup memory limit is
    well under the host's RAM, long before the GPU is touched. Pointing torch's default
    device at the GPU for the duration of __init__ makes every nn.Linear allocate
    straight into VRAM: peak host RAM stays near zero, and the trailing .to() is a no-op.

    Passing `dtype` also redirects torch's default dtype, halving the build's peak VRAM
    (~14 GB fp32 -> ~7 GB bf16). Off by default because it is not numerically free:
    anything computed at construction lands in bf16 rather than fp32. Parameters do not
    care (the pretrained load overwrites every one), but a buffer derived at __init__ —
    RoPE inverse frequencies being the classic case — would keep the reduced precision.
    transformers guards inv_freq with an explicit .float(), so bf16 is believed safe;
    pass dtype="bfloat16" below only if the fp32 build genuinely does not fit."""
    want_cuda = str(device).startswith("cuda") and torch.cuda.is_available()
    if not want_cuda:
        yield
        return
    prev = torch.get_default_dtype()
    if dtype is not None:
        torch.set_default_dtype({"bfloat16": torch.bfloat16, "float16": torch.float16,
                                 "float32": torch.float32}[str(dtype).lower()])
    try:
        with torch.device(device):
            yield
    finally:
        torch.set_default_dtype(prev)


def pick_build_dtype(cfg_dtype, device, tight_gb=24.0):
    """Decide whether to construct in reduced precision, from FREE VRAM.

    Constructing pi0/pi05 in fp32 costs ~21 GB transiently before lerobot casts to
    cfg.dtype. That is fine on an empty 40 GB+ card and is numerically the safest
    path, so it stays the default. But free VRAM is often far below total: a crashed
    kernel keeps its CUDA context and allocations alive, and inside a container those
    processes usually cannot be signalled (nvidia-smi reports host PIDs you have no
    permission over). Rather than die with a CUDA OOM that reads as "model too big
    for this GPU", drop the build to cfg.dtype (~7 GB) when the headroom isn't there.

    Safe because every parameter is overwritten by the pretrained load moments later;
    only construction-time buffers keep the reduced precision, and transformers guards
    the one that matters (RoPE inv_freq) with an explicit .float()."""
    if not (str(device).startswith("cuda") and torch.cuda.is_available()):
        return None
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    if free_gb >= tight_gb:
        return None
    print(f"only {free_gb:.1f} GB VRAM free (< {tight_gb:.0f} GB) — constructing in "
          f"{cfg_dtype} (~7 GB) instead of fp32 (~21 GB). If that's unexpected, a dead "
          f"process is holding VRAM; restarting the pod is the clean fix.")
    return cfg_dtype


def quantize_vlm(policy, mode, lora_rank=0, expert_only=False,
                 compute_dtype=torch.bfloat16):
    """QLoRA-style k-bit quantization of the FROZEN VLM, in place. No-op when off.

    Swaps every nn.Linear inside the PaliGemma tower for a bitsandbytes Linear4bit
    (NF4 + double quantization) or Linear8bitLt, moving each to the GPU IMMEDIATELY —
    bitsandbytes quantizes lazily on .cuda(), so going layer-by-layer frees each
    full-precision weight as we go and keeps the peak at one layer instead of a
    second copy of the model.

    NF4 = 4-bit normal-float storage with bf16 dequantization at matmul time: the 2B
    VLM drops ~4.6 GB -> ~1.4 GB, which is what buys back the headroom for a larger
    batch. Throughput stays close to bf16; the cost is a small quality hit that the
    LoRA adapters largely absorb.

    Deliberately NOT quantized:
      • the action expert — it is FULLY trained, and 4-bit weights take no gradient,
        so quantizing it would silently freeze the one part that must learn;
      • lm_head / embeddings — vocabulary-sized and weight-tied;
      • norms — 1-D, negligible memory, and most damaged by quantization."""
    mode = (mode or "none").lower()
    if mode in ("none", "off", ""):
        return
    assert mode in ("nf4", "int8"), f"QUANTIZE must be none|nf4|int8, got {mode!r}"
    if lora_rank <= 0 and not expert_only:
        raise ValueError(
            f"quantize={mode!r} freezes the VLM (4-bit weights take no gradient) but "
            f"vlm_lora_rank=0 and train_expert_only=False — nothing would train.")
    import bitsandbytes as bnb

    vlm  = policy.model.paligemma_with_expert.paligemma
    skip = ("lm_head", "embed_tokens", "embed_out")
    n_swapped = saved = 0

    def _swap(module, prefix=""):
        nonlocal n_swapped, saved
        for name, child in list(module.named_children()):
            path = f"{prefix}.{name}" if prefix else name
            if isinstance(child, nn.Linear) and not any(s in path for s in skip):
                w = child.weight.data
                b = child.bias.data if child.bias is not None else None
                if mode == "nf4":
                    new = bnb.nn.Linear4bit(
                        child.in_features, child.out_features, bias=b is not None,
                        compute_dtype=compute_dtype, quant_type="nf4",
                        compress_statistics=True)          # double quantization
                    new.weight = bnb.nn.Params4bit(
                        w.to(compute_dtype), requires_grad=False,
                        quant_type="nf4", compress_statistics=True)
                else:
                    new = bnb.nn.Linear8bitLt(
                        child.in_features, child.out_features, bias=b is not None,
                        has_fp16_weights=False, threshold=6.0)
                    new.weight = bnb.nn.Int8Params(
                        w.to(torch.float16), requires_grad=False, has_fp16_weights=False)
                if b is not None:
                    new.bias = nn.Parameter(b.to(compute_dtype), requires_grad=False)
                # move NOW — this is where bnb quantizes, and it lets the
                # full-precision weight be freed before the next layer is built.
                setattr(module, name, new.to("cuda"))
                n_swapped += 1
                saved += w.numel() * (w.element_size() - (0.5 if mode == "nf4" else 1))
                del w, b, child
            else:
                _swap(child, path)

    _swap(vlm)
    torch.cuda.empty_cache()

    # QLoRA + gradient checkpointing: the checkpointed blocks sit behind a fully frozen
    # base, so without an input that requires grad the recomputed graph is detached and
    # the adapters get NO gradient — training silently does nothing.
    if getattr(policy.config, "gradient_checkpointing", False):
        try:
            vlm.enable_input_require_grads()
        except AttributeError:
            print("WARNING: could not enable input grads on the VLM; if LoRA grads "
                  "come back None, set gradient_checkpointing False")

    print(f"quantized VLM to {mode.upper()}: {n_swapped} Linear layers, "
          f"~{saved/1e9:.1f} GB saved (action expert + lm_head left in bf16)")

class PiPolicy(nn.Module):
    """π0 wrapper — mean-std norm in the wrapper, IDENTITY inside lerobot."""

    def __init__(self, cfg: PolicyCfg, stats: dict, device=None):
        super().__init__()
        self.cfg = cfg
        self.image_keys = [f"observation.images.{c}" for c in cfg.camera_names]
        # Order is load -> quantize -> LoRA, and it is not interchangeable:
        # quantization is destructive (NF4-ing random weights and then loading over
        # Params4bit does not round-trip), and peft must see Linear4bit to build
        # lora.Linear4bit wrappers.
        with build_context(device, pick_build_dtype(cfg.dtype, device)):
            self.policy = PI0Policy(_lerobot_config(cfg))
        if cfg.pretrained:
            _load_pretrained_weights(self.policy, cfg.pretrained)
        else:
            print("WARNING: random-init weights — smoke tests only, NOT finetuning")
        quantize_vlm(self.policy, cfg.quantize, cfg.vlm_lora_rank, cfg.train_expert_only)
        if cfg.vlm_lora_rank > 0:
            _inject_vlm_lora(self.policy, cfg.vlm_lora_rank,
                             cfg.vlm_lora_alpha, cfg.vlm_lora_dropout)
        self.tokenizer = AutoTokenizer.from_pretrained("google/paligemma-3b-pt-224")
        for k in ("state_mean", "state_std", "action_mean", "action_std"):
            self.register_buffer(k, torch.as_tensor(stats[k]).float())
        self.register_buffer("_imagenet_mean", _IMN_MEAN.clone())
        self.register_buffer("_imagenet_std",  _IMN_STD.clone())

    def _norm_state(self, s):    return (s - self.state_mean) / self.state_std
    def _norm_action(self, a):   return (a - self.action_mean) / self.action_std
    def _unnorm_action(self, a): return a * self.action_std + self.action_mean
    def _to_raw(self, img):      # undo ImageNet norm -> [0,1]; lerobot maps to [-1,1]
        return (img * self._imagenet_std + self._imagenet_mean).clamp(0, 1)

    def _make_batch(self, obs_state, actions=None, action_is_pad=None,
                    obs_image=None, task=None, training=None):
        if training is None:
            training = self.training
        B = obs_state.shape[0]
        task = task or [TASK_TEXT] * B
        state = mask_state(self._norm_state(obs_state), self.cfg.proprio_mode,
                           self.cfg.proprio_dropout_rate, training)
        batch = {STATE_KEY: state}                     # (B, state_dim) — no seq dim
        if self.cfg.use_image and obs_image is not None:
            if torch.is_tensor(obs_image):                 # 1 cam -> dict
                obs_image = {self.image_keys[0]: obs_image}
            for _k in self.image_keys:                     # feed each camera
                batch[_k] = self._to_raw(obs_image[_k])
        enc = self.tokenizer(list(task), return_tensors="pt", padding="max_length",
                             truncation=True, max_length=self.cfg.tokenizer_max_length)
        batch[LANG_TOKENS]    = enc["input_ids"].to(obs_state.device)
        batch[LANG_ATTN_MASK] = enc["attention_mask"].to(obs_state.device)
        if actions is not None:
            batch[ACTION_KEY]      = self._norm_action(actions)
            batch["action_is_pad"] = action_is_pad
        return batch

    def forward(self, obs_state, actions, action_is_pad, obs_image=None, task=None):
        loss, _ = self.policy.forward(
            self._make_batch(obs_state, actions, action_is_pad, obs_image, task))
        return loss, loss.item(), 0.0

    def reset(self):
        self.policy.reset()

    @torch.no_grad()
    def predict(self, obs_state, obs_image=None, task=None):
        return self._unnorm_action(self.policy.select_action(
            self._make_batch(obs_state, obs_image=obs_image, task=task, training=False)))


def build_model(cfg: dict, stats: dict, device):
    m, d = cfg["model"], cfg["dataset"]
    return PiPolicy(PolicyCfg(
        state_dim=m["state_dim"], action_dim=m["action_dim"],
        chunk_size=d["chunk_size"], use_image=d["use_image"],
        camera_names=tuple(d.get("camera_names", ("wrist_cam", "scene_cam"))),
        tokenizer_max_length=m["tokenizer_max_length"],
        pretrained=m.get("pretrained", ""),
        vlm_lora_rank=m.get("vlm_lora_rank", 0),
        vlm_lora_alpha=m.get("vlm_lora_alpha", 32),
        vlm_lora_dropout=m.get("vlm_lora_dropout", 0.05),
        dtype=m["dtype"], gradient_checkpointing=m["gradient_checkpointing"],
        freeze_vision_encoder=m["freeze_vision_encoder"],
        train_expert_only=m["train_expert_only"],
        quantize=m.get("quantize", "none") or "none",
        proprio_mode=m["proprio_mode"],
        proprio_dropout_rate=m["proprio_dropout_rate"]),
        stats, device=device).to(device)

print("π0 wrapper defined")

## 8 · Assemble the config

Mirrors the repo's `policies/pi0/config.yaml` schema — this dict is stored inside
every checkpoint, which is what lets the repo's `deploy.py` rebuild the model on the robot box.

In [ ]:
CFG = {
    "dataset": {"root": DATA_DIR, "chunk_size": CHUNK_SIZE, "use_image": True,
                "image_size": [224, 224], "val_frac": VAL_FRAC, "aug_level": AUG_LEVEL,
                "camera_names": CAMERA_NAMES},
    "model": {"state_dim": STATE_DIM, "action_dim": ACTION_DIM,
              "paligemma_variant": "gemma_2b", "action_expert_variant": "gemma_300m",
              "max_state_dim": 32, "max_action_dim": 32,
              "tokenizer_max_length": 48, "num_inference_steps": 10,
              "pretrained": PRETRAINED,
              "vlm_lora_rank": LORA_RANK if FINETUNE_MODE == "lora" else 0,
              "vlm_lora_alpha": LORA_ALPHA, "vlm_lora_dropout": LORA_DROPOUT,
              "dtype": "bfloat16", "gradient_checkpointing": True,
              "quantize": QUANTIZE,
              "freeze_vision_encoder": FINETUNE_MODE == "freeze_vision",
              "train_expert_only":     FINETUNE_MODE == "expert_only",   # lora mode freezes the VLM via vlm_lora_rank
              "proprio_mode": PROPRIO_MODE, "proprio_dropout_rate": 0.3},
    "training": {"batch_size": BATCH_SIZE, "lr": LR, "weight_decay": WEIGHT_DECAY,
                 "max_epochs": MAX_EPOCHS, "grad_clip": GRAD_CLIP,
                 "save_every": SAVE_EVERY, "log_every": LOG_EVERY, "checkpoint_dir": CKPT_DIR,
                 "device": "cuda", "seed": SEED},
}
import json
print(json.dumps({"finetune": {k: CFG["model"][k] for k in
      ("pretrained", "vlm_lora_rank", "dtype", "quantize", "gradient_checkpointing",
       "freeze_vision_encoder", "train_expert_only")},
      "training": CFG["training"]}, indent=2))

## 9 · Build datasets + model, time one step

Downloads the pretrained π0 weights (~6 GB, first run only) and confirms they load. Prints trainable/frozen parameter counts
and times one forward+backward — if this OOMs, fix it **now** (cell 1: `expert_only`
/ smaller batch, then re-run cells 1 → 4 → 8 → 9), not 20 minutes into training.
The model built here is reused by the training loop.

In [ ]:
import time, torch
from torch.utils.data import DataLoader

torch.manual_seed(SEED)

n_eps = json.loads((pathlib.Path(DATA_DIR) / "meta/info.json").read_text())["total_episodes"]
train_eps, val_eps = FR5Dataset.episode_split(n_eps, VAL_FRAC, SEED)
train_ds = FR5Dataset(DATA_DIR, CHUNK_SIZE, episode_indices=train_eps, aug_level=AUG_LEVEL, frame_stride=FRAME_STRIDE)
val_ds   = FR5Dataset(DATA_DIR, CHUNK_SIZE, episode_indices=val_eps,   aug_level="none", frame_stride=FRAME_STRIDE)
stats    = train_ds.get_stats()
print(f"train: {len(train_eps)} episodes / {len(train_ds)} samples   "
      f"val: {len(val_eps)} episodes / {len(val_ds)} samples")

model = build_model(CFG, stats, torch.device("cuda"))
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"trainable {trainable/1e6:.0f}M   frozen {frozen/1e6:.0f}M   ({FINETUNE_MODE})")

b = torch.utils.data.default_collate([train_ds[i] for i in range(BATCH_SIZE)])
t0 = time.time()
loss, _, _ = model(b["observation.state"].cuda(), b["action"].cuda(),
                   b["action_is_pad"].cuda(),
                   {k: b[k].cuda() for k in b if k.startswith("observation.images.")},
                   task=list(b["task"]))
loss.backward(); torch.cuda.synchronize()
step_s = time.time() - t0
spe = max(1, len(train_ds) // BATCH_SIZE)
print(f"1 step = {step_s:.2f}s  →  ~{step_s*spe/60:.1f} min/epoch, "
      f"~{step_s*spe*MAX_EPOCHS/3600:.1f} h for {MAX_EPOCHS} epochs")
print(f"peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f} / {vram:.0f} GB")
model.zero_grad(set_to_none=True)

## 10 · Train

Same loop semantics as the repo's `common/train.py`: AdamW over `requires_grad` params,
grad-norm clipping, per-epoch validation, `metrics.csv`, periodic + `best.pt` checkpoints
in the **repo's checkpoint format** (so `deploy.py --checkpoint best.pt` just works).

> If the browser disconnects, the Jupyter **kernel keeps running server-side** on RunPod —
> reopen the notebook and check progress via cell 11 (outputs of this cell are lost, the
> run is not).

In [ ]:
import csv, time, pathlib, torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

ckpt_dir = pathlib.Path(CKPT_DIR); ckpt_dir.mkdir(parents=True, exist_ok=True)
# num_workers=0: RunPod containers default to a tiny /dev/shm; DataLoader
# workers pass image tensors through it and crash the kernel (SIGBUS).
# Training is GPU-bound so single-process loading costs ~nothing. Raise this
# only if you launched the pod with a large --shm-size (e.g. 16g).
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True)
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),
                              lr=LR, weight_decay=WEIGHT_DECAY)

# Two granularities on purpose. metrics.csv stays one row per epoch — it is what
# the monitor cell plots and the only place val loss exists, since validation runs
# per epoch. metrics_steps.csv is the fine-grained view: with ~4000 steps/epoch, an
# epoch-only curve hides a divergence for hours, and on a run this slow that is the
# difference between killing it early and wasting a day.
metrics_path = ckpt_dir / "metrics.csv"
steps_path   = ckpt_dir / "metrics_steps.csv"
if not metrics_path.exists():
    with open(metrics_path, "w", newline="") as f:
        csv.writer(f).writerow(["epoch", "train_l1", "val_l1", "train_val_gap",
                                "grad_norm", "lr", "seconds"])
if not steps_path.exists():
    with open(steps_path, "w", newline="") as f:
        csv.writer(f).writerow(["step", "epoch", "train_l1", "grad_norm", "lr", "samples_per_s"])

global_step = 0

def run_epoch(loader, train, epoch=0):
    global global_step
    model.train(train)
    tot, gn, n = 0.0, 0.0, 0
    t_win, n_win, loss_win, gn_win = time.time(), 0, 0.0, 0.0   # LOG_EVERY window
    with (torch.enable_grad() if train else torch.no_grad()):
        pbar = tqdm(loader, leave=False, desc="train" if train else "val")
        for batch in pbar:
            loss, li, _ = model(batch["observation.state"].cuda(),
                                batch["action"].cuda(), batch["action_is_pad"].cuda(),
                                {k: batch[k].cuda() for k in batch if k.startswith("observation.images.")},
                                task=list(batch["task"]))
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                g = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP).item()
                optimizer.step()
                global_step += 1
                gn += g
                loss_win += li; gn_win += g; n_win += 1
                if global_step % LOG_EVERY == 0:
                    _l, _g = loss_win / n_win, gn_win / n_win
                    _sps = n_win * BATCH_SIZE / max(time.time() - t_win, 1e-9)
                    # append + close each time rather than holding a handle: this run
                    # has died mid-epoch before, and buffered rows would be lost.
                    with open(steps_path, "a", newline="") as _f:
                        csv.writer(_f).writerow([global_step, epoch, f"{_l:.6f}",
                                                 f"{_g:.4f}", LR, f"{_sps:.1f}"])
                    if run:
                        run.log({"step/loss": _l, "step/grad_norm": _g, "step/lr": LR,
                                 "step/samples_per_s": _sps, "epoch": epoch},
                                step=global_step)
                    pbar.set_postfix(loss=f"{_l:.4f}", gn=f"{_g:.2f}", sps=f"{_sps:.1f}")
                    t_win, n_win, loss_win, gn_win = time.time(), 0, 0.0, 0.0
            tot += li; n += 1
    return tot / max(n, 1), gn / max(n, 1)

run = None
if USE_WANDB:
    run = wandb.init(project=WANDB_PROJECT,
                     name=f"{POLICY}-{FINETUNE_MODE}-bs{BATCH_SIZE}",
                     config={**CFG["model"], **CFG["training"],
                             "policy": POLICY, "finetune_mode": FINETUNE_MODE,
                             "dataset_repo": HF_DATASET_REPO,
                             "train_episodes": len(train_eps), "val_episodes": len(val_eps)})

# Push during training, not only at the end. The upload cell at the bottom of this
# notebook only runs if you actually reach it — a kernel death at epoch 87 leaves 87
# epochs of work on a pod whose disk does not outlive the pod. Uploads run in a
# background thread so a multi-GB transfer doesn't stall the GPU.
push_repo, _push_fut = None, None
if PUSH_EVERY:
    from huggingface_hub import HfApi, whoami
    _api = HfApi()
    push_repo = f"{whoami()['name']}/fr5-{POLICY}-{FINETUNE_MODE}"
    _api.create_repo(push_repo, private=True, exist_ok=True)
    print(f"periodic push -> https://huggingface.co/{push_repo}  (every {PUSH_EVERY} epochs)")

best_val = float("inf")
for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()
    train_l1, grad_norm = run_epoch(train_loader, True, epoch)
    val_l1, _           = run_epoch(val_loader, False)
    secs = time.time() - t0
    with open(metrics_path, "a", newline="") as f:
        csv.writer(f).writerow([epoch, f"{train_l1:.6f}", f"{val_l1:.6f}",
                                f"{val_l1 - train_l1:.6f}", f"{grad_norm:.4f}",
                                LR, f"{secs:.1f}"])
    if run:
        run.log({"train/loss": train_l1, "val/loss": val_l1,
                 "val/gap": val_l1 - train_l1, "train/grad_norm": grad_norm,
                 "train/lr": LR, "epoch_seconds": secs, "epoch": epoch},
                step=global_step)
    is_best = val_l1 < best_val
    best_val = min(best_val, val_l1)
    print(f"epoch {epoch:3d}/{MAX_EPOCHS}  train {train_l1:.4f}  val {val_l1:.4f}"
          f"{'  ↑ best' if is_best else ''}  ({secs:.0f}s)")
    if is_best or epoch % SAVE_EVERY == 0 or epoch == MAX_EPOCHS:
        ckpt = {"epoch": epoch, "policy": POLICY, "model_state": model.state_dict(),
                "val_l1": val_l1, "config": CFG, "stats": stats,
                "action_space": train_ds.info.get("action_space", "joint")}
        if epoch % SAVE_EVERY == 0 or epoch == MAX_EPOCHS:
            torch.save(ckpt, ckpt_dir / f"epoch_{epoch:04d}.pt")
        if is_best:
            torch.save(ckpt, ckpt_dir / "best.pt")
    if push_repo and (epoch % PUSH_EVERY == 0 or epoch == MAX_EPOCHS):
        if _push_fut is not None and not _push_fut.done():
            # best.pt is multi-GB; if the previous transfer is still going, stacking
            # another would just contend for bandwidth. Skip — the next one catches up.
            print(f"   push skipped @ epoch {epoch}: previous upload still in flight")
        else:
            _push_fut = _api.upload_folder(
                folder_path=CKPT_DIR, repo_id=push_repo,
                allow_patterns=["best.pt", "metrics.csv", "metrics_steps.csv"],
                commit_message=f"epoch {epoch} · val_l1 {val_l1:.4f}",
                run_as_future=True)
            print(f"   pushing epoch {epoch} (val {val_l1:.4f}) -> {push_repo} in background")

if _push_fut is not None:
    print("waiting for the last background upload...")
    _push_fut.result()          # block here, not in the loop, so nothing is lost on exit
if run:
    run.summary["best_val_l1"] = best_val
print(f"done — best val_l1 {best_val:.4f}   checkpoints in {ckpt_dir}")

## 11 · Monitor — re-run any time (also after a browser reconnect)

In [ ]:
import pandas as pd, pathlib
import matplotlib.pyplot as plt

steps_path = pathlib.Path(CKPT_DIR, "metrics_steps.csv")
if steps_path.exists():
    sm = pd.read_csv(steps_path)
    fig, ax = plt.subplots(1, 3, figsize=(16, 3.0))
    ax[0].plot(sm.step, sm.train_l1, lw=0.7, alpha=0.4, color="tab:blue")
    # rolling mean over ~an epoch's worth of logged points: the raw trace is noisy
    # enough that a real trend change is easy to miss by eye.
    if len(sm) > 20:
        ax[0].plot(sm.step, sm.train_l1.rolling(max(5, len(sm)//50), min_periods=1).mean(),
                   lw=1.6, color="tab:blue")
    ax[0].set_title("train loss / step"); ax[0].set_xlabel("step")
    ax[1].plot(sm.step, sm.grad_norm, lw=0.8, color="tab:orange")
    ax[1].set_title("grad norm / step"); ax[1].set_xlabel("step")
    ax[2].plot(sm.step, sm.samples_per_s, lw=0.8, color="tab:green")
    ax[2].set_title("throughput (samples/s)"); ax[2].set_xlabel("step")
    for a in ax: a.grid(alpha=0.25)
    plt.tight_layout(); plt.show()
    print(f"{len(sm)} step-logs · last {sm.step.iloc[-1]} steps · "
          f"mean {sm.samples_per_s.mean():.1f} samples/s")
else:
    print("metrics_steps.csv not written yet — appears after the first LOG_EVERY steps")

csv_path = pathlib.Path(CKPT_DIR, "metrics.csv")
if csv_path.exists():
    m = pd.read_csv(csv_path)
    fig, ax = plt.subplots(1, 4, figsize=(16, 3.2))
    m.plot(x="epoch", y=["train_l1", "val_l1"], ax=ax[0], title="flow-matching loss")
    m.plot(x="epoch", y="train_val_gap", ax=ax[1], title="train/val gap (overfitting watch)")
    m.plot(x="epoch", y="grad_norm", ax=ax[2], title="grad norm")
    m.plot(x="epoch", y="seconds", ax=ax[3], title="seconds / epoch")
    plt.tight_layout(); plt.show()
    best = m.loc[m.val_l1.idxmin()]
    print(f"best val_l1 = {best.val_l1:.4f} @ epoch {int(best.epoch)}  ({len(m)} epochs logged)")
else:
    print("metrics.csv not written yet — appears after epoch 1")

## 12 · Ground-truth vs prediction — the real quality check

Rolls the trained policy over one **validation** episode (open-loop chunks, like the
repo's eval layer) and overlays predicted joint trajectories on the ground truth.
Numbers lie less than loss curves: look for tracking through the grasp region.
Logged to wandb as a figure when enabled.

In [ ]:
import torch
import matplotlib.pyplot as plt

model.eval(); model.reset()
gt, pred, last_ep = [], [], None
with torch.no_grad():
    for i in range(len(val_ds)):
        ep_idx = val_ds._samples[i][0]
        if ep_idx != last_ep:          # new episode -> clear the action-chunk queue
            model.reset(); last_ep = ep_idx
        item = val_ds[i]
        gt.append(item["action"][0])
        a = model.predict(item["observation.state"].unsqueeze(0).cuda(),
                          obs_image={k: item[k].unsqueeze(0).cuda() for k in item if k.startswith("observation.images.")},
                          task=[item["task"]])
        pred.append(a.squeeze(0).float().cpu())
gt, pred = torch.stack(gt), torch.stack(pred)

D = gt.shape[1]                                   # action dim (auto, not hardcoded)
names = ([f"joint{j+1}" for j in range(6)] + ["gripper"]) if D == 7 else [f"a{j}" for j in range(D)]
mae = (gt - pred).abs().mean(0)
ncols = 4
nrows = -(-(D + 1) // ncols)                      # ceil: one panel per dim + MAE summary
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
axes = axes.flatten()
for j in range(D):
    axes[j].plot(gt[:, j], label="ground truth", lw=1.5)
    axes[j].plot(pred[:, j], label="prediction", lw=1.0, alpha=0.85)
    axes[j].set_title(names[j]); axes[j].legend(fontsize=7)
axes[D].axis("off")
axes[D].text(0, 0.5, "MAE per dim:\n" +
             "\n".join(f"{n}: {v:.3f}" for n, v in zip(names, mae)), fontsize=9)
for k in range(D + 1, len(axes)):
    axes[k].axis("off")
plt.suptitle(f"{POLICY} — GT vs prediction, validation episodes ({len(val_ds)} steps)")
plt.tight_layout(); plt.show()
print("mean MAE:", f"{mae.mean():.4f}")

if USE_WANDB and wandb.run:
    wandb.log({"eval/gt_vs_pred": wandb.Image(fig), "eval/mae_mean": mae.mean().item()})
    wandb.finish()

## 12b · Replay — camera view + prediction, side by side

Rolls the policy forward one frame at a time (exactly as deployment does, action queue and all) and animates what the model saw next to what it predicted, with a playhead sweeping the trajectory. Jerk and lag show up here in a way a static plot hides. Cost is one full ODE solve per frame — lower `VIZ_MAX_FRAMES` if it drags.

In [ ]:
# ── replay: camera view + GT-vs-prediction, side by side, as a video ──────────
VIZ_EPISODE    = None   # None -> first validation episode
VIZ_MAX_FRAMES = 240    # cost knob: pi0 runs a full ODE solve per frame
VIZ_FPS        = 15
VIZ_DIMS       = None   # None -> every action dim; or e.g. [0, 1, 2] for joints 1-3

import numpy as np, torch, matplotlib.pyplot as plt
from matplotlib import animation, gridspec
from IPython.display import HTML, display
from tqdm.auto import tqdm

ep = int(val_eps[0]) if VIZ_EPISODE is None else int(VIZ_EPISODE)
# aug_level="none" and frame_stride=1: augmentation would show the model a different
# image than the one we draw, and striding would desync lerobot's action queue —
# select_action pops one action per call, so frames must be consumed consecutively.
viz_ds = FR5Dataset(DATA_DIR, chunk_size=CHUNK_SIZE, episode_indices=[ep],
                    aug_level="none", frame_stride=1)
n    = min(len(viz_ds), VIZ_MAX_FRAMES)
cams = viz_ds.camera_keys
print(f"episode {ep}: {len(viz_ds)} frames, replaying {n} · cameras {cams}")

model.eval(); model.reset()          # reset = clear the action-chunk queue
imgs, gt, pred = [], [], []
with torch.no_grad():
    for i in tqdm(range(n), desc="rolling out"):
        item = viz_ds[i]
        a = model.predict(item["observation.state"].unsqueeze(0).cuda(),
                          obs_image={k: item[k].unsqueeze(0).cuda() for k in cams},
                          task=[item["task"]])
        pred.append(a.squeeze(0).float().cpu().numpy())
        gt.append(item["action"][0].numpy())
        # undo the ImageNet norm so we draw what the model actually saw
        imgs.append(np.hstack([model._to_raw(item[k]).permute(1, 2, 0).numpy() for k in cams]))

gt, pred = np.asarray(gt), np.asarray(pred)
err  = np.abs(gt - pred).mean(1)
dims = list(range(gt.shape[1])) if VIZ_DIMS is None else list(VIZ_DIMS)
names = ([f"joint{i+1}" for i in range(6)] + ["gripper"]) if gt.shape[1] == 7 \
        else [f"a{i}" for i in range(gt.shape[1])]

fig = plt.figure(figsize=(14, 5.2), dpi=80)
gs  = gridspec.GridSpec(2, 2, width_ratios=[1.15, 1], height_ratios=[2.2, 1],
                        hspace=0.3, wspace=0.18)
ax_img  = fig.add_subplot(gs[:, 0]); ax_img.axis("off")
ax_traj = fig.add_subplot(gs[0, 1])
ax_err  = fig.add_subplot(gs[1, 1])

im = ax_img.imshow(imgs[0])
ax_img.set_title(" | ".join(c.split(".")[-1] for c in cams), fontsize=10)

colors = plt.cm.tab10(np.linspace(0, 1, 10))
for d in dims:                        # ground truth solid, prediction dashed
    ax_traj.plot(gt[:, d],   color=colors[d % 10], lw=1.4, label=names[d])
    ax_traj.plot(pred[:, d], color=colors[d % 10], lw=1.2, ls="--", alpha=0.85)
ax_traj.set_ylabel("action"); ax_traj.set_title("— ground truth   -- predicted", fontsize=10)
ax_traj.legend(fontsize=7, ncol=4, loc="upper right"); ax_traj.grid(alpha=0.25)

ax_err.plot(err, color="crimson", lw=1.3)
ax_err.fill_between(range(n), err, color="crimson", alpha=0.18)
ax_err.set_ylabel("mean |err|"); ax_err.set_xlabel("frame"); ax_err.grid(alpha=0.25)

heads = [a.axvline(0, color="k", lw=1.2, alpha=0.7) for a in (ax_traj, ax_err)]
txt = ax_traj.text(0.01, 0.04, "", transform=ax_traj.transAxes, fontsize=9,
                   family="monospace", bbox=dict(fc="white", alpha=0.75, ec="none"))

def _update(i):
    im.set_data(imgs[i])
    for h in heads:
        h.set_xdata([i, i])
    txt.set_text(f"frame {i:3d}   |err| {err[i]:.4f}")
    return [im, *heads, txt]

anim = animation.FuncAnimation(fig, _update, frames=n,
                               interval=1000 / VIZ_FPS, blit=False)
plt.close(fig)

print(f"mean |err| over the episode: {err.mean():.4f}   worst frame: {err.argmax()} ({err.max():.4f})")
try:                       # real <video> if ffmpeg is on the pod, else a JS player
    display(HTML(anim.to_html5_video()))
except Exception:
    display(HTML(anim.to_jshtml()))


## 13 · Ship checkpoints to the Hub

Uploads `best.pt` + `metrics.csv` to a private model repo so they survive pod termination.
On the robot box: download `best.pt` and run `python common/deploy.py --checkpoint best.pt`.

In [ ]:
from huggingface_hub import HfApi, whoami

repo = f"{whoami()['name']}/fr5-{POLICY}-{FINETUNE_MODE}"
api = HfApi()
api.create_repo(repo, private=True, exist_ok=True)
api.upload_folder(folder_path=CKPT_DIR, repo_id=repo,
                  allow_patterns=["best.pt", "metrics.csv", "metrics_steps.csv"],
                  commit_message=f"{POLICY} {FINETUNE_MODE} bs={BATCH_SIZE} epochs={MAX_EPOCHS}")
print(f"uploaded -> https://huggingface.co/{repo}")

## Troubleshooting

**CUDA OOM** — in order of preference:
1. `FINETUNE_MODE = "expert_only"` (freezes the 2B VLM; the 300M expert still learns the task)
2. halve `BATCH_SIZE` (set it explicitly, e.g. `BATCH_SIZE = 1`)
3. both

Both live in **cell 1 (Parameters)** — edit there, restart the kernel (frees VRAM cleanly),
then run cells **1 → 9** again and relaunch cell 10.

**`GatedRepoError` / 403 on PaliGemma** — the HF account hasn't accepted the license, or the token lacks read scope.

**Pod / kernel restarted mid-run** — checkpoints land every `SAVE_EVERY` (10) epochs plus `best.pt`.
There's no optimizer-state resume; a relaunch of cell 10 restarts from scratch weights, so for
spot pods prefer shorter `MAX_EPOCHS` per run and upload checkpoints (cell 12) as you go.

**Throughput sanity** — cell 9 prints measured step time, min/epoch, full-run hours, and peak VRAM.
Watch the first epochs for a *decreasing* train loss before walking away.